<a href="https://colab.research.google.com/github/miftahulhdd/MiftahulHudaAmri_2411533005_ML2526/blob/main/BoostingVoting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score

In [9]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

boosting

In [10]:
#adaboost

ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train_scaled, y_train)

acc_ada = accuracy_score(y_test, ada.predict(X_test_scaled))
print("AdaBoost Accuracy:", acc_ada)

AdaBoost Accuracy: 0.9736842105263158


In [11]:
#gradient boosting

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_scaled, y_train)

acc_gb = accuracy_score(y_test, gb.predict(X_test_scaled))
print("Gradient Boosting Accuracy:", acc_gb)

Gradient Boosting Accuracy: 0.956140350877193


voting classifier

In [12]:
#voting / soft voting

log = LogisticRegression(max_iter=5000)
rf = RandomForestClassifier(n_estimators=100)
gb = GradientBoostingClassifier(n_estimators=100)

voting = VotingClassifier(
    estimators=[('log', log), ('rf', rf), ('gb', gb)],
    voting='soft'
)

voting.fit(X_train_scaled, y_train)

acc_vote = accuracy_score(y_test, voting.predict(X_test_scaled))
print("Voting Accuracy:", acc_vote)

Voting Accuracy: 0.9649122807017544


hyperparameter tuning

In [13]:
# gridsearch random forest

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(),
    param_grid,
    cv=5
)

grid.fit(X_train_scaled, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'max_depth': None, 'n_estimators': 100}
Best CV Score: 0.9626373626373625


pca+ensemble

In [14]:
pca = PCA(n_components=10)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

gb_pca = GradientBoostingClassifier(n_estimators=100)
gb_pca.fit(X_train_pca, y_train)

acc_gb_pca = accuracy_score(y_test, gb_pca.predict(X_test_pca))
print("Gradient Boosting + PCA:", acc_gb_pca)

Gradient Boosting + PCA: 0.9649122807017544


---

In [15]:
# bantu isi tabel

# 1. Melatih dan mengevaluasi Model Tunggal (Single Model) untuk kebutuhan tabel
log_single = LogisticRegression(max_iter=5000, random_state=42)
rf_single = RandomForestClassifier(n_estimators=100, random_state=42)

log_single.fit(X_train_scaled, y_train)
rf_single.fit(X_train_scaled, y_train)

acc_log = accuracy_score(y_test, log_single.predict(X_test_scaled))
acc_rf = accuracy_score(y_test, rf_single.predict(X_test_scaled))

# 2. Mengambil akurasi dari Best Tuned Model (menggunakan objek 'grid' dari GridSearchCV)
acc_tuned_rf = accuracy_score(y_test, grid.predict(X_test_scaled))

# 3. Mengompilasi seluruh hasil ke dalam DataFrame
tabel_data = {
    'Model': [
        'Logistic',
        'Random Forest',
        'AdaBoost',
        'Gradient Boosting',
        'Voting',
        'Best Tuned Model',
        'Boosting + PCA'
    ],
    'Accuracy': [
        acc_log,         # Hasil evaluasi Logistic Regression tunggal
        acc_rf,          # Hasil evaluasi Random Forest tunggal
        acc_ada,         # Mengambil nilai dari sel AdaBoost (0.9737)
        acc_gb,          # Mengambil nilai dari sel Gradient Boosting (0.9561)
        acc_vote,        # Mengambil nilai dari sel Voting Classifier (0.9649)
        acc_tuned_rf,    # Hasil akurasi dari model RF yang sudah di-tuning
        acc_gb_pca       # Mengambil nilai dari sel GB + PCA (0.9649)
    ]
}

df_kesimpulan = pd.DataFrame(tabel_data)

# 4. Menampilkan tabel akhir dengan format 4 angka di belakang koma
print("=" * 43)
print("       TABEL PERBANDINGAN AKHIR MODEL      ")
print("=" * 43)
print(df_kesimpulan.to_string(index=False, formatters={'Accuracy': '{:,.4f}'.format}))
print("=" * 43)

       TABEL PERBANDINGAN AKHIR MODEL      
            Model Accuracy
         Logistic   0.9737
    Random Forest   0.9649
         AdaBoost   0.9737
Gradient Boosting   0.9561
           Voting   0.9649
 Best Tuned Model   0.9649
   Boosting + PCA   0.9649
